# Lane Detection: U-Net (ResNet34) with Entropy Minimization
## Overview
This notebook implements a lane detection system developed as part of an **Automated Driving License Testing System**, a *BSc capstone project*. Using the MoLane dataset (part of the CARLANE benchmark), it aims to narrow the gap between simulated training environments and real-world testing tracks through semantic segmentation and Unsupervised Domain Adaptation (UDA).
## Model Architecture
The core of the system is a U-Net architecture paired with a ResNet34 encoder (pre-trained on ImageNet).
**Hybrid Training Strategy:** It uses **Supervised Learning** *(Baseline training on labeled source data)* and **Unsupervised Domain Adaptation**(UDA) to improve generalization on target-domain images, implementing Entropy Minimization.
## Getting Started
By default, this notebook is configured for a quick run (though “quick” may be optimistic,that one epoch can take close to 2 hours, atleast that is what it took for me) with `NUM_EPOCHS = 1`. For final-level weights or competitive leaderboard performance, it is recommended to set `NUM_EPOCHS = 50` in the Setup & Configuration section.
## Links
https://www.kaggle.com/code/aelafgetaneh/lane-detection-u-net-with-entropy-minimization

## 1. Setup and configuration

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

# Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# Paths
DATA_ROOT = "/kaggle/input/datasets/carlanebenchmark/carlane-benchmark/CARLANE/MoLane/data"
SPLITS_ROOT = "/kaggle/input/datasets/carlanebenchmark/carlane-benchmark/CARLANE/MoLane/splits"

# Image size
IMG_HEIGHT = 360
IMG_WIDTH = 640

# Training parameters
BATCH_SIZE = 8
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

## 2. Load split files

In [ ]:
def load_split_txt(file_path, has_labels=True):
    """
    Reads a Kaggle split text file.

    Labeled line format:
        image_path mask_path ...

    Unlabeled line format:
        image_path
    """
    samples = []
    with open(file_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue

            if has_labels:
                samples.append((parts[0], parts[1]))
            else:
                samples.append(parts[0])
    return samples


source_train_txt = os.path.join(SPLITS_ROOT, "source_train.txt")
source_val_txt = os.path.join(SPLITS_ROOT, "source_val.txt")
target_train_txt = os.path.join(SPLITS_ROOT, "target_train.txt")
target_val_txt = os.path.join(SPLITS_ROOT, "target_val.txt")
target_test_txt = os.path.join(SPLITS_ROOT, "target_test.txt")

source_train_samples = load_split_txt(source_train_txt, has_labels=True)
source_val_samples = load_split_txt(source_val_txt, has_labels=True)
target_train_samples = load_split_txt(target_train_txt, has_labels=False)
target_val_samples = load_split_txt(target_val_txt, has_labels=True)
target_test_samples = load_split_txt(target_test_txt, has_labels=True)

print(f"Source train: {len(source_train_samples)}")
print(f"Source val: {len(source_val_samples)}")
print(f"Target train (unlabeled): {len(target_train_samples)}")
print(f"Target val: {len(target_val_samples)}")
print(f"Target test: {len(target_test_samples)}")

## 3. Dataset class

In [ ]:
class LaneSegmentationDataset(Dataset):
    def __init__(self, root_dir, samples, transform=None, is_labeled=True):
        self.root_dir = root_dir
        self.samples = samples
        self.transform = transform
        self.is_labeled = is_labeled

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.is_labeled:
            img_path, label_path = self.samples[idx]
        else:
            img_path = self.samples[idx]
            label_path = None

        img_full_path = os.path.join(self.root_dir, img_path)
        img = cv2.imread(img_full_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_full_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if label_path is not None:
            mask_full_path = os.path.join(self.root_dir, label_path)
            mask = cv2.imread(mask_full_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                raise FileNotFoundError(f"Mask not found: {mask_full_path}")
            mask = (mask > 0).astype(np.uint8)
        else:
            mask = None

        if self.transform:
            if mask is not None:
                augmented = self.transform(image=img, mask=mask)
                img = augmented["image"]
                mask = augmented["mask"]
            else:
                augmented = self.transform(image=img)
                img = augmented["image"]

        # Albumentations + ToTensorV2 usually gives a tensor already
        if not isinstance(img, torch.Tensor):
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        if mask is not None:
            if isinstance(mask, torch.Tensor):
                mask = mask.long()
            else:
                mask = torch.from_numpy(mask).long()
        else:
            mask = torch.zeros(IMG_HEIGHT, IMG_WIDTH, dtype=torch.long)

        # Remove extra channel if any
        if isinstance(mask, torch.Tensor) and mask.dim() == 3 and mask.shape[0] == 1:
            mask = mask.squeeze(0)

        return img, mask

## 4. Applying Transforms

In [ ]:
train_transform = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

## 5. Datasets and dataloaders

In [ ]:
source_train_dataset = LaneSegmentationDataset(DATA_ROOT, source_train_samples, transform=train_transform, is_labeled=True)
source_val_dataset = LaneSegmentationDataset(DATA_ROOT, source_val_samples, transform=val_transform, is_labeled=True)
target_train_dataset = LaneSegmentationDataset(DATA_ROOT, target_train_samples, transform=train_transform, is_labeled=False)
target_val_dataset = LaneSegmentationDataset(DATA_ROOT, target_val_samples, transform=val_transform, is_labeled=True)
target_test_dataset = LaneSegmentationDataset(DATA_ROOT, target_test_samples, transform=val_transform, is_labeled=True)

source_train_loader = DataLoader(source_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
source_val_loader = DataLoader(source_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
target_train_loader = DataLoader(target_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
target_val_loader = DataLoader(target_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
target_test_loader = DataLoader(target_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 6. Model definition

In [ ]:
class UNetWithResNetEncoder(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.n_classes = n_classes

        try:
            resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        except AttributeError:
            resnet = models.resnet34(pretrained=True)

        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.enc2 = nn.Sequential(resnet.layer1)
        self.enc3 = resnet.layer2
        self.enc4 = resnet.layer3
        self.enc5 = resnet.layer4

        self.up5 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec5 = self._make_decoder_block(256 + 256, 256)

        self.up4 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec4 = self._make_decoder_block(128 + 128, 128)

        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec3 = self._make_decoder_block(64 + 64, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.final = nn.Conv2d(32, n_classes, kernel_size=1)

    def _make_decoder_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    @staticmethod
    def _crop_to_match(tensor, target):
        _, _, h, w = tensor.shape
        th, tw = target.shape[2], target.shape[3]
        return tensor[:, :, :th, :tw]

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        d5 = self.up5(e5)
        d5 = self._crop_to_match(d5, e4)
        d5 = torch.cat([d5, e4], dim=1)
        d5 = self.dec5(d5)

        d4 = self.up4(d5)
        d4 = self._crop_to_match(d4, e3)
        d4 = torch.cat([d4, e3], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = self._crop_to_match(d3, e2)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)

        out = self.up2(d3)
        out = self.final(out)
        return out


model = UNetWithResNetEncoder(n_classes=2).to(DEVICE)
print(model)

## 7. Loss, optimizer, helpers

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(dataloader, desc="Training")

    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        if masks.dim() == 4 and masks.shape[1] == 1:
            masks = masks.squeeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(len(dataloader), 1)


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validation"):
            images = images.to(device)
            masks = masks.to(device)
            if masks.dim() == 4 and masks.shape[1] == 1:
                masks = masks.squeeze(1)

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / max(len(dataloader), 1)


def entropy_loss(logits):
    probs = torch.softmax(logits, dim=1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
    return entropy.mean()


def compute_iou(preds, targets, num_classes=2):
    ious = []
    for cls in range(num_classes):
        pred_inds = (preds == cls)
        target_inds = (targets == cls)
        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()
        if union == 0:
            iou = torch.tensor(float("nan"), device=preds.device)
        else:
            iou = intersection / union
        ious.append(iou)
    return ious


def evaluate_model_iou(model, loader, device):
    model.eval()
    total_iou = 0.0
    num_samples = 0

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Evaluating"):
            images = images.to(device)
            masks = masks.to(device)
            if masks.dim() == 4 and masks.shape[1] == 1:
                masks = masks.squeeze(1)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            ious = compute_iou(preds, masks, num_classes=2)
            lane_iou = ious[1]
            if torch.isnan(lane_iou):
                lane_iou = torch.tensor(0.0, device=device)

            total_iou += lane_iou.item() * images.size(0)
            num_samples += images.size(0)

    return total_iou / max(num_samples, 1)

## 8. Supervised training on source domain

In [ ]:
best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    train_loss = train_one_epoch(model, source_train_loader, optimizer, criterion, DEVICE)
    val_loss = validate(model, source_val_loader, criterion, DEVICE)
    scheduler.step()

    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_source_model.pth")
        print("Saved best source model.")

## 9. Target validation before adaptation

In [ ]:
source_only_model = UNetWithResNetEncoder(n_classes=2).to(DEVICE)
source_only_model.load_state_dict(torch.load("best_source_model.pth", map_location=DEVICE))
target_val_loss_before = validate(source_only_model, target_val_loader, criterion, DEVICE)
print(f"Target Validation Loss (source-only model): {target_val_loss_before:.4f}")

## 10. Unsupervised domain adaptation with entropy minimization

In [ ]:
model = UNetWithResNetEncoder(n_classes=2).to(DEVICE)
model.load_state_dict(torch.load("best_source_model.pth", map_location=DEVICE))

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE * 0.5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

alpha = 0.1
best_adapt_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    model.train()
    total_sup_loss = 0.0
    total_ent_loss = 0.0

    num_batches = min(len(source_train_loader), len(target_train_loader))
    source_iter = iter(source_train_loader)
    target_iter = iter(target_train_loader)

    pbar = tqdm(range(num_batches), desc=f"Adaptation Epoch {epoch + 1}/{NUM_EPOCHS}")
    for _ in pbar:
        try:
            src_images, src_masks = next(source_iter)
        except StopIteration:
            source_iter = iter(source_train_loader)
            src_images, src_masks = next(source_iter)

        try:
            tgt_images, _ = next(target_iter)
        except StopIteration:
            target_iter = iter(target_train_loader)
            tgt_images, _ = next(target_iter)

        src_images = src_images.to(DEVICE)
        src_masks = src_masks.to(DEVICE)
        if src_masks.dim() == 4 and src_masks.shape[1] == 1:
            src_masks = src_masks.squeeze(1)

        tgt_images = tgt_images.to(DEVICE)

        optimizer.zero_grad()

        src_logits = model(src_images)
        sup_loss = criterion(src_logits, src_masks)

        tgt_logits = model(tgt_images)
        ent_loss = entropy_loss(tgt_logits)

        loss = sup_loss + alpha * ent_loss
        loss.backward()
        optimizer.step()

        total_sup_loss += sup_loss.item()
        total_ent_loss += ent_loss.item()

        pbar.set_postfix(
            sup_loss=f"{sup_loss.item():.4f}",
            ent_loss=f"{ent_loss.item():.4f}",
            total_loss=f"{loss.item():.4f}"
        )

    print(f"Epoch {epoch + 1}: Sup Loss = {total_sup_loss / max(num_batches, 1):.4f}, "
          f"Entropy Loss = {total_ent_loss / max(num_batches, 1):.4f}")

    target_val_loss = validate(model, target_val_loader, criterion, DEVICE)
    print(f"Target Val Loss = {target_val_loss:.4f}")

    if target_val_loss < best_adapt_val_loss:
        best_adapt_val_loss = target_val_loss
        torch.save(model.state_dict(), "best_adapted_model.pth")
        print("Saved best adapted model.")

    scheduler.step()

model.load_state_dict(torch.load("best_adapted_model.pth", map_location=DEVICE))
print("Loaded best adapted model.")

## 11. Test both source and adapted models and choose the best one

In [ ]:
source_model = UNetWithResNetEncoder(n_classes=2).to(DEVICE)
source_model.load_state_dict(torch.load("best_source_model.pth", map_location=DEVICE))

adapted_model = UNetWithResNetEncoder(n_classes=2).to(DEVICE)
adapted_model.load_state_dict(torch.load("best_adapted_model.pth", map_location=DEVICE))

source_test_iou = evaluate_model_iou(source_model, target_test_loader, DEVICE)
adapted_test_iou = evaluate_model_iou(adapted_model, target_test_loader, DEVICE)

print(f"Target Test IoU (source-only): {source_test_iou:.4f}")
print(f"Target Test IoU (adapted):     {adapted_test_iou:.4f}")

if adapted_test_iou >= source_test_iou:
    final_model = adapted_model
    torch.save(final_model.state_dict(), "final_lane_model.pth")
    print("Selected adapted model as final model.")
else:
    final_model = source_model
    torch.save(final_model.state_dict(), "final_lane_model.pth")
    print("Selected source-only model as final model.")

## 12. Visualize predictions

In [ ]:
def visualize_prediction(model, dataloader, device, num_samples=4):
    model.eval()
    images, masks = next(iter(dataloader))
    images = images[:num_samples].to(device)
    masks = masks[:num_samples]

    if masks.dim() == 4 and masks.shape[1] == 1:
        masks = masks.squeeze(1)

    masks = masks.cpu().numpy()

    with torch.no_grad():
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

    images = images.cpu().numpy().transpose(0, 2, 3, 1)

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    images = images * std + mean
    images = np.clip(images, 0, 1)

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 3))
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(num_samples):
        axes[i, 0].imshow(images[i])
        axes[i, 0].set_title("Image")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(masks[i], cmap="gray")
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(preds[i], cmap="gray")
        axes[i, 2].set_title("Prediction")
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()

visualize_prediction(final_model, target_test_loader, DEVICE, num_samples=4)

## 13. Save final model and inference note

In [ ]:
torch.save(final_model.state_dict(), "final_lane_model.pth")
print("Final model saved as final_lane_model.pth")